In [14]:
from fractions import Fraction
from typing import Tuple, Union
import numpy as np
import pandas as pd
from sympy import Matrix
from IPython.display import display

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.float_format', '{:.15f}'.format)  # Ép hiển thị dạng thập phân chuẩn
pd.set_option('display.width', 300)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp lặp Gauss - Seidel

# Thuật toán
**Input**: 
- $A_{n \times n}$ và $B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$

**Output**: Nghiệm xấp xỉ $X^*$ của hệ phương trình

1. Tính $q=\max\limits_{i=\overline{1,n}}\displaystyle\sum\limits_{\substack{i=1 \\ j \neq i}}^n \dfrac{|a_{ij}|}{|a_{ii}|}$ 

    Nếu $q<1$ thì $p=\infty, \mu = \max\limits_{i=\overline{1,n}}\dfrac{\displaystyle\sum\limits_{j=1}^{i-1}|a_{ij}|}{|a_{ii}|-\displaystyle\sum\limits_{j=i+1}^{n}|a_{ij}|}$ và $s=0$ và qua bước 3

2. Tính $q=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{\substack{j=1 \\ i \neq j}}^n \dfrac{|a_{ij}|}{|a_{jj}|}$. 

    Nếu $q \geq 1 \to$ dừng chương trình

    Trái lại:  $p=1, \mu = \max\limits_{j=\overline{1,n}}\dfrac{\displaystyle\sum\limits_{i=1}^{j-1}|a_{ij}|}{|a_{jj}|-\displaystyle\sum\limits_{i=j+1}^{n}|a_{ij}|}$ và $s=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{i=j+1}^{n} \dfrac{|a_{ij}|}{|a_{jj}|}$

3. Khởi tạo $T=\text{diag}\left(\dfrac{1}{a_{11}}, \dfrac{1}{a_{22}},\ldots,\dfrac{1}{a_{nn}}\right)$

4. Tính $C=I-TA$ và $D=TB$

5. Tách $C=L+U$, với $l_{ij} = \begin{cases} c_{ij} & i \geq j \\ 0 & i<j \end{cases} $ và $u_{ij} = \begin{cases} 0 & i>j \\ c_{ij} & i \leq j \end{cases}$

6. Khởi tạo $X_0=[0]_{n \times m}$, khởi tạo $k=1$, tính $\varepsilon_0 = \dfrac{(1-\mu)(1-s)}{\mu}\varepsilon$ hoặc $\eta_0 = \dfrac{(1-\mu)(1-s)}{\mu}\eta$. 
7. Lặp:
    - Tính $X_{k} = L \cdot X_{k} + U \cdot X_{k-1} + D$
    - Tính $\Delta_k = \|X_{k} - X_{k-1}\|_p$ hoặc $\delta_k=\dfrac{\|X_{k}-X_{k-1}\|_p}{\|X_{k}\|_p}$ 

    Lặp cho đến khi $\Delta_k \leq \varepsilon_0$ hoặc $\delta_k \leq \eta_0$
8. In ra $X^*$ là nghiệm xấp xỉ của hệ phương trình.

# Thuật toán đánh giá sai số tiên nghiệm (Gauss-Seidel)

**Input**: 
- $A_{n \times n}$ và $B_{n \times m}$
- Sai số tuyệt đối $\varepsilon$ hoặc sai số tương đối $\eta$

**Output**: Nghiệm xấp xỉ $X^*$ của hệ phương trình

1. Tính $q=\max\limits_{i=\overline{1,n}}\displaystyle\sum\limits_{\substack{i=1 \\ j \neq i}}^n \dfrac{|a_{ij}|}{|a_{ii}|}$. 

    Nếu $q<1$ thì $p=\infty, \mu = \max\limits_{i=\overline{1,n}}\dfrac{\displaystyle\sum\limits_{j=1}^{i-1}|a_{ij}|}{|a_{ii}|-\displaystyle\sum\limits_{j=i+1}^{n}|a_{ij}|}$ và $s=0$ và qua bước 3.

2. Tính $q=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{\substack{j=1 \\ i \neq j}}^n \dfrac{|a_{ij}|}{|a_{jj}|}$. 

    Nếu $q \ge 1 \to$ dừng chương trình. 
    Trái lại: $p=1, \mu = \max\limits_{j=\overline{1,n}}\dfrac{\displaystyle\sum\limits_{i=1}^{j-1}|a_{ij}|}{|a_{jj}|-\displaystyle\sum\limits_{i=j+1}^{n}|a_{ij}|}$ và $s=\max\limits_{j=\overline{1,n}}\displaystyle\sum\limits_{i=j+1}^{n} \dfrac{|a_{ij}|}{|a_{jj}|}$.

3. Khởi tạo $T=\text{diag}\left(\dfrac{1}{a_{11}}, \dfrac{1}{a_{22}},\ldots,\dfrac{1}{a_{nn}}\right)$

4. Tính $C=I-TA$ và $D=TB$.

5. Tách $C=L+U$, với $l_{ij} = \begin{cases} c_{ij} & i \ge j \\ 0 & i<j \end{cases} $ và $u_{ij} = \begin{cases} 0 & i>j \\ c_{ij} & i \le j \end{cases}$

6. Khởi tạo $X_0=[0]_{n \times m}$. Tính $X_1 = L \cdot X_1 + U \cdot X_0 + D$. 

7. Đặt $\Delta_1 = \|X_1 - X_0\|_p$. Nếu $\Delta_1 = 0 \Rightarrow$ Trả về $X^* = X_0$ và dừng thuật toán.

8. Tính $\varepsilon_0 = \dfrac{(1-\mu)(1-s)}{\mu}\varepsilon$ (nếu dùng sai số tuyệt đối) hoặc $\eta_0 = \dfrac{(1-\mu)(1-s)}{\mu}\eta$ (nếu dùng sai số tương đối).
    Thiết lập khoảng sai số $\delta = \varepsilon_0$ (hoặc $\delta = \eta_0 \|X_1\|_p$).

9. Tính số bước lặp tiên nghiệm $N$:
    - Nếu $\Delta_1 \le \delta \Rightarrow$ Gán $N = 1$.
    - Trái lại $\Rightarrow$ Gán $N = \left\lceil \dfrac{\ln(\delta / \Delta_1)}{\ln(\mu)} \right\rceil$.

10. Lặp $k$ từ $2$ đến $N$:
    - Tính $X_{k} = L \cdot X_{k} + U \cdot X_{k-1} + D$

11. In ra $X^* = X_N$ là nghiệm xấp xỉ của hệ phương trình.

In [15]:
def input_matrix(filename, convert_fractions=False):
    """
    Reads a matrix from a text file and returns it as a NumPy array.
    Supports fractional entries if present (though your files use decimals).
    """
    matrix = []

    with open(filename, 'r') as f:
        for line in f:
            tokens = line.strip().split()
            if not tokens:
                continue

            row = []
            for token in tokens:
                if '/' in token:
                    # convert fractions if any includes fraction sign
                    val = Fraction(token)
                    row.append(float(val) if convert_fractions else val)
                else:
                    # parse as float directly
                    row.append(float(token))

            matrix.append(row)
            
    dtype = float if convert_fractions else object
    return np.array(matrix, dtype=dtype)


In [16]:
def output_matrix(X: np.ndarray, precision: int = 15):
    """
    Prints a NumPy array (vector or matrix) in LaTeX mathematical format using SymPy.
    
    Parameters:
    - X: np.ndarray, 1D or 2D array.
    - precision: number of decimal places to round floats to.
    """
    # Render natively as a LaTeX matrix in Jupyter
    # Ép kiểu về float để tránh lỗi np.round với mảng dtype=object
    X_float = np.array(X, dtype=float)
    display(Matrix(np.round(X_float, precision)))

In [17]:
def check_dominance(A: np.ndarray) -> int:
    """
    Returns:
      1 if A is row-dominant only,
      2 if A is column-dominant only,
      3 if both,
      0 if neither.
    """
    n = A.shape[0]
    row_dom = all(abs(A[i,i]) > np.sum(np.abs(A[i,:])) - abs(A[i,i]) for i in range(n))
    col_dom = all(abs(A[j,j]) > np.sum(np.abs(A[:,j])) - abs(A[j,j]) for j in range(n))
    if row_dom and col_dom:
        return 3
    if row_dom:
        return 1
    if col_dom:
        return 2
    return 0

domi_dict = {0: 'Không chéo trội', 1: 'Chéo trội hàng', 2: 'Chéo trội cột', 3: 'Chéo trội cả hàng và cột'}

In [18]:
def convert_to_iteration_row(A: np.ndarray, B: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """Converts Ax = B into x_new = C x + D for row-dominant case."""
    T = np.diag(1.0 / np.diag(A))
    n = A.shape[0]
    C = np.eye(n) - T.dot(A)
    D = T.dot(B)
    return C, D

def convert_to_iteration_col(A: np.ndarray, B: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Prepares y_{k+1} = C y_k + D for column-dominant case."""
    T = np.diag(1.0 / np.diag(A))
    n = A.shape[0]
    C = np.eye(n) - A.dot(T)
    D = B.copy()
    return T, C, D


In [19]:
def gauss_seidel_iteration(
    A: np.ndarray,
    B: np.ndarray,
    x0,
    domiType: int,
    eps=None,
    eta=None,
    mode='posteriori',
    max_iterations=1000
):
    if (eps is None) == (eta is None):
        raise ValueError("Specify exactly one of eps (exact) or eta (relative)")
    
    is_matrix = True if np.array(x0).ndim == 2 and np.array(x0).shape[1] > 1 else False

    if domiType in (1, 3):
        C, D = convert_to_iteration_row(A, B)
        
        print("\nMa trận lặp C:")
        output_matrix(C)
        print("\nVector/Ma trận lặp D:")
        output_matrix(D)
        
        if is_matrix:
            vec_norm = lambda x: np.linalg.norm(x, ord=np.inf)
        else:
            vec_norm = lambda x: np.max(np.abs(x))
        
        n = C.shape[0]
        L = np.tril(C, k=-1)
        U = np.triu(C, k=1)

        row_L = np.sum(np.abs(L), axis=1)
        row_U = np.sum(np.abs(U), axis=1)
        ratios = row_L / (1 - row_U)
        q = np.max(ratios)
        s = 0.0
        
        tol = (eps if eps is not None else eta) * (1 - q) * (1 - s) / q
        print(f"Sử dụng logic chéo trội hàng. q: {q:.7f}, s: {s:.7f}, Ngưỡng sai số (tol): {tol:.7f}")
        print(f"Chế độ lặp: {mode.upper()}")

        x_old = np.array(x0, dtype=float)
        
        if mode == 'apriori':
            x1 = np.zeros_like(x_old)
            for k in range(n):
                x1[k] = (L[k].dot(x1)) + (U[k].dot(x_old)) + D[k]
            diff_1 = vec_norm(x1 - x_old)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_diff = tol if eps is not None else tol * vec_norm(x1)
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"Số bước lặp tiên nghiệm: N = {max_iterations}")

        if not is_matrix:
            x_old = x_old.flatten()
            history = [x_old.copy()]
            errors = [float('nan')]

            for k_iter in range(1, max_iterations + 1):
                x_new = np.zeros_like(x_old)
                for k in range(n):
                    x_new[k] = (L[k].dot(x_new)) + (U[k].dot(x_old)) + D[k]

                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                history.append(x_new.copy())
                errors.append(float(err))
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    break
            else:
                if mode == 'posteriori':
                    print(f"Cảnh báo: Không hội tụ sau {max_iterations} vòng lặp.")
                else:
                    print(f"Đã hoàn thành {max_iterations} vòng lặp tiên nghiệm.")
            
            n_vars = len(x_new)
            import pandas as pd
            df = pd.DataFrame(history, columns=[f"x{i+1}" for i in range(n_vars)])
            df["error"] = errors
            df.index.name = "k"
            df_T = df.T
            df_T.index.name = "X_k"
            return df_T, x_new.reshape(-1, 1)
        else:
            print("X_0 =")
            output_matrix(x_old)
            for k_iter in range(1, max_iterations + 1):
                x_new = np.zeros_like(x_old)
                for k in range(n):
                    x_new[k] = (L[k].dot(x_new)) + (U[k].dot(x_old)) + D[k]

                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                print(f"\n--- Vòng lặp k = {k_iter} ---")
                print(f"X_{k_iter} =")
                output_matrix(x_new)
                err_str = f"{err:.4e}" if err < 1e-4 else f"{err:.6f}"
                print(f"Sai số Delta_{k_iter} = {err_str}")
                
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    print(f"\n==> Hội tụ sau {k_iter} bước lặp (sai số <= {tol:.4e}).")
                    break
            else:
                if mode == 'posteriori':
                    print(f"\n==> Đạt giới hạn lặp tối đa {max_iterations} bước.")
                else:
                    print(f"\n==> Đã hoàn thành {max_iterations} bước lặp theo công thức tiên nghiệm.")
            return None, x_old

    elif domiType == 2:
        T, C, D = convert_to_iteration_col(A, B)

        print("\nMa trận lặp C:")
        output_matrix(C)
        print("\nVector/Ma trận lặp D:")
        output_matrix(D)
        print("\nMa trận T:")
        output_matrix(T)

        if is_matrix:
            vec_norm = lambda x: np.linalg.norm(x, ord=1)
        else:
            vec_norm = lambda x: np.sum(np.abs(x))
        
        n = C.shape[0]
        L = np.tril(C, k=-1)
        U = np.triu(C, k=1)

        col_L = np.sum(np.abs(L), axis=0)
        col_U = np.sum(np.abs(U), axis=0)
        ratios = col_U / (1 - col_L)
        q = np.max(ratios)
        s = np.max(col_L)

        tol = (eps if eps is not None else eta) * (1 - q) * (1 - s) / q
        print(f"Sử dụng logic chéo trội cột. q: {q:.7f}, s: {s:.7f}, Ngưỡng sai số (tol): {tol:.7f}")
        print(f"Chế độ lặp: {mode.upper()}")

        T_inv = np.diag(1.0 / np.diag(T))
        x_old = np.array(x0, dtype=float)
        y_old = T_inv.dot(x_old)
        
        if mode == 'apriori':
            y1 = y_old.copy()
            for k in range(n):
                y1[k] = L[k].dot(y1) + U[k].dot(y_old) + D[k]
            x1 = T.dot(y1)
            diff_1 = vec_norm(x1 - x_old)
            if diff_1 == 0:
                print("\n==> X_1 = X_0, nghiệm đã hội tụ ngay từ đầu.")
                max_iterations = 0
            else:
                target_diff = tol if eps is not None else tol * vec_norm(x1)
                if diff_1 <= target_diff:
                    max_iterations = 1
                else:
                    n_steps = np.ceil(np.log(target_diff / diff_1) / np.log(q))
                    max_iterations = int(n_steps) if n_steps > 0 else 0
                print(f"Số bước lặp tiên nghiệm: N = {max_iterations}")
        
        if not is_matrix:
            x_old = x_old.flatten()
            y_old = T_inv.dot(x_old)
            
            history = [np.concatenate([y_old, x_old])]
            errors = [float('nan')]

            for k_iter in range(1, max_iterations + 1):
                y_new = y_old.copy()
                for k in range(n):
                    y_new[k] = L[k].dot(y_new) + U[k].dot(y_old) + D[k]

                x_new = T.dot(y_new)
                
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                history.append(np.concatenate([y_new, x_new]))
                errors.append(float(err))
                y_old = y_new
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    break
            else:
                if mode == 'posteriori':
                    print(f"Cảnh báo: Không hội tụ sau {max_iterations} vòng lặp.")
                else:
                    print(f"Đã hoàn thành {max_iterations} vòng lặp tiên nghiệm.")
                
            n_vars = len(x_new)
            import pandas as pd
            df = pd.DataFrame(history, columns=[f"y{i+1}" for i in range(n_vars)] + [f"x{i+1}" for i in range(n_vars)])
            df["error"] = errors
            df.index.name = "k"
            df_T = df.T
            df_T.index.name = "X_k"
            return df_T, x_new.reshape(-1, 1)
        else:
            y_old = T_inv.dot(x_old)
            print("X_0 =")
            output_matrix(x_old)
            for k_iter in range(1, max_iterations + 1):
                y_new = y_old.copy()
                for k in range(n):
                    y_new[k] = L[k].dot(y_new) + U[k].dot(y_old) + D[k]

                x_new = T.dot(y_new)
                
                err = vec_norm(x_new - x_old)
                if eta is not None:
                    err /= vec_norm(x_new)

                print(f"\n--- Vòng lặp k = {k_iter} ---")
                print(f"X_{k_iter} =")
                output_matrix(x_new)
                err_str = f"{err:.4e}" if err < 1e-4 else f"{err:.6f}"
                print(f"Sai số Delta_{k_iter} = {err_str}")
                
                y_old = y_new
                x_old = x_new
                if mode == 'posteriori' and tol is not None and err <= tol:
                    print(f"\n==> Hội tụ sau {k_iter} bước lặp (sai số <= {tol:.4e}).")
                    break
            else:
                if mode == 'posteriori':
                    print(f"\n==> Đạt giới hạn lặp tối đa {max_iterations} bước.")
                else:
                    print(f"\n==> Đã hoàn thành {max_iterations} bước lặp theo công thức tiên nghiệm.")
            return None, x_old
            
    else:
        raise ValueError("Ma trận không chéo trội, phương pháp Gauss-Seidel có thể không hội tụ.")

def solve_gauss_seidel_system(A, B, eps=1e-6, eta=None, mode='posteriori', max_iterations=1000):
    domiType = check_dominance(A)
    if domiType == 0:
        print("Lỗi: Ma trận không chéo trội. Thuật toán tự động kết thúc.")
        return None

    if B.ndim == 1 or B.shape[1] == 1:
        B_mat = B.flatten()
        x0 = np.zeros_like(B_mat)
        df_history, X_final = gauss_seidel_iteration(A, B_mat, x0, domiType, eps, eta, mode, max_iterations)
        display(df_history)
        print("\nNghiệm gần đúng:")
        output_matrix(X_final)
        return X_final
    else:
        B_mat = B
        x0 = np.zeros_like(B_mat)
        print("\n--- GIẢI CHO HỆ VỚI VẾ PHẢI LÀ MA TRẬN ---")
        _, X_final = gauss_seidel_iteration(A, B_mat, x0, domiType, eps, eta, mode, max_iterations)
        print("\n=> KẾT LUẬN NGHIỆM TỔNG QUÁT DƯỚI DẠNG MA TRẬN:")
        print("Hệ có nghiệm X =")
        output_matrix(X_final)
        return X_final


### Kết quả

In [20]:
# Test với ma trận GS_input_A và GS_input_B
A = input_matrix('G_input_A.txt', convert_fractions=True)
B = input_matrix('G_input_B.txt', convert_fractions=True)

print("\nMa trận A:")
output_matrix(A)
print("\nKiểm tra chéo trội của A:", domi_dict[check_dominance(A)])
print("\nMa trận B:")
output_matrix(B)

# Giải hệ
eps = 1e-10
X = solve_gauss_seidel_system(A, B, eps=eps)


Ma trận A:


Matrix([
[ 131.75,    3.75,    5.25,    11.25,   18.25,     6.5,    12.0,     15.5,    2.75,     20.0],
[   -7.0,   157.0,     4.0,      8.0, 17.6667, -3.6667,    -3.0,      4.0,     9.0,  21.6667],
[    2.8,     4.2,    90.0,      5.4,    10.8,    12.6,    -7.8,      8.0,    14.8,     13.0],
[16.5714, 14.8571, 25.4286, 141.4286, 25.7143,  2.1429,  5.5714,  10.5714,  5.4286,   6.4286],
[ 6.1667,   -19.0,  9.8333,  12.1667,   115.0, -6.1667,    14.5,   3.6667,   -10.5, -14.1667],
[ -0.875,  11.125,   7.125,    6.375,  13.375,   82.75,    -6.0,    1.875,  -15.75,    -8.75],
[-7.5714, -1.4286,  5.2857,   9.8571,   -18.0,  7.1429, 97.8571, -10.1429, -8.8571, -12.8571],
[   17.2,    11.0,    15.6,      2.8,     2.2,    14.2,    15.6,    107.8,     5.2,      6.4],
[15.8333,  4.8333, -5.8333,      5.5, -9.6667,  7.6667,  5.6667,      9.5, 82.1667,     -3.0],
[ 0.9167,    -2.0, -2.5833,   2.9167,  3.9167, 10.6667,  2.1667,   9.5833, -3.5833,  53.0833]])


Kiểm tra chéo trội của A: Chéo trội hàng

Ma trận B:


Matrix([
[ 310.2275],
[-193.5166],
[  238.112],
[-165.0229],
[ 214.4215],
[ 108.6325],
[  28.8543],
[  291.732],
[-256.5717],
[ 197.1857]])


Ma trận lặp C:


...

Matrix([
[  2.3546679316888],
[ -1.2325898089172],
[ 2.64568888888889],
[-1.16682835013569],
[  1.8645347826087],
[ 1.31277945619335],
[0.294861588990477],
[ 2.70623376623377],
[-3.12257520382345],
[ 3.71464660260383]])

Sử dụng logic chéo trội hàng. q: 0.8170478, s: 0.0000000, Ngưỡng sai số: 0.0000000


k,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
X_k,,,,,,,,,,,,,,,
x1,0.000000000000000,2.354667931688805,1.516500022785175,1.516468964350103,1.519095920866734,1.519152281358003,1.519058934582479,1.519070384729932,1.519069842261453,1.519069925246655,1.519069903786347,1.519069906739415,1.519069906399570,1.519069906448148,1.519069906440948
x2,0.000000000000000,-1.127604614510690,-1.526778825055808,-1.533193705939452,-1.524595825037195,-1.524684505038155,-1.524788512008179,-1.524778305267876,-1.524778240302174,-1.524778239268276,-1.524778253707197,-1.524778251630735,-1.524778251809334,-1.524778251784978,-1.524778251789068
x3,0.000000000000000,2.625054101913514,2.466321160219360,2.473194086370036,2.474689413750411,2.474305158581724,2.474285088964556,2.474285878584465,2.474286598738719,2.474286549246466,2.474286549541153,2.474286548800805,2.474286548937030,2.474286548924987,2.474286548926739
x4,0.000000000000000,-1.796253094358984,-2.087856025475408,-2.119387094432824,-2.117317290610114,-2.116854538713286,-2.116938046458144,-2.116936507010960,-2.116936596793950,-2.116936495359266,-2.116936510180752,-2.116936508772767,-2.116936508978207,-2.116936508946099,-2.116936508950435
x5,0.000000000000000,1.517547909676025,1.507869184797504,1.488500884498258,1.487609467684367,1.488177730217110,1.488138643318062,1.488140070517586,1.488139535800230,1.488139641820078,1.488139629620076,1.488139631189574,1.488139630954617,1.488139630987406,1.488139630982836
x6,0.000000000000000,1.156348465077305,0.911563854545902,0.988349022901040,0.989098676679626,0.988471802184698,0.988510263598349,0.988511552127530,0.988511426992281,0.988511380328516,0.988511389276219,0.988511388574669,0.988511388636670,0.988511388623804,0.988511388625962
x7,0.000000000000000,0.694464843721999,0.944796294854614,0.987911208324651,0.982958248339042,0.983000705055937,0.983012884716774,0.983016773172936,0.983015944550223,0.983016020210447,0.983016012422649,0.983016013751656,0.983016013544075,0.983016013572365,0.983016013568526
x8,0.000000000000000,1.828586473483083,2.019712619152047,1.988716339586078,1.988525649400447,1.988672218501780,1.988693564195375,1.988689351009596,1.988689601902475,1.988689572491718,1.988689578635716,1.988689577778571,1.988689577892832,1.988689577876354,1.988689577878592
x9,0.000000000000000,-3.392058876583568,-3.109638335297188,-3.114321898688663,-3.115493427303004,-3.115440526281382,-3.115423350087549,-3.115426182998067,-3.115426041412952,-3.115426048578961,-3.115426045246837,-3.115426045856816,-3.115426045789783,-3.115426045798047,-3.115426045796768



Nghiệm gần đúng:


Matrix([
[ 1.51906990644095],
[-1.52477825178907],
[ 2.47428654892674],
[-2.11693650895043],
[ 1.48813963098284],
[0.988511388625962],
[0.983016013568526],
[ 1.98868957787859],
[-3.11542604579677],
[ 2.94980791644024]])